In [ ]:
### first 25%dataset

import os
import shutil
import zipfile
import csv
from google.colab import drive
from huggingface_hub import hf_hub_download

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Local working directories (FAST local Colab storage)
local_base = "/content/processed_dataset"
local_low = os.path.join(local_base, "sentinaiplow")
local_high = os.path.join(local_base, "sentinaiphigh")
local_meta = os.path.join(local_base, "metadata")

os.makedirs(local_low, exist_ok=True)
os.makedirs(local_high, exist_ok=True)
os.makedirs(local_meta, exist_ok=True)

# 3. Target Drive Folder
drive_target_folder = "/content/drive/MyDrive/Project_Folder/Datasets"
os.makedirs(drive_target_folder, exist_ok=True)

# 4. Download
print("Downloading SEN2NAIP cross-sensor zip...")
zip_path = hf_hub_download(
    repo_id="isp-uv-es/SEN2NAIP",
    repo_type="dataset",
    filename="cross-sensor/cross-sensor.zip",
    local_dir="/content"
)

# 5. Extract
extract_path = "/content/sen2naip_extracted"
print("Extracting files locally...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# 6. Process Folders locally
base_extracted_folder = os.path.join(extract_path, "cross-sensor")
if not os.path.exists(base_extracted_folder):
    base_extracted_folder = extract_path

rois = sorted([d for d in os.listdir(base_extracted_folder) if d.startswith("ROI_")])
print(f"Total regions: {len(rois)}. Processing 100% of regions locally...")

metadata_file = os.path.join(local_meta, "sen2naip_metadata.csv")
with open(metadata_file, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(["Region_ID", "LR_Image_Path", "HR_Image_Path", "Dataset_Source"])

    for roi in rois:
        roi_folder = os.path.join(base_extracted_folder, roi)

        lr_actual = next((f for f in os.listdir(roi_folder) if f.lower() == 'lr.tif'), None)
        hr_actual = next((f for f in os.listdir(roi_folder) if f.lower() == 'hr.tif'), None)

        if lr_actual and hr_actual:
            lr_src = os.path.join(roi_folder, lr_actual)
            hr_src = os.path.join(roi_folder, hr_actual)

            lr_name = f"{roi}_lr.tif"
            hr_name = f"{roi}_hr.tif"

            # Copy locally
            shutil.copy(lr_src, os.path.join(local_low, lr_name))
            shutil.copy(hr_src, os.path.join(local_high, hr_name))

            # CSV path entries
            writer.writerow([
                roi,
                f"Datasets/sentinaiplow/{lr_name}",
                f"Datasets/sentinaiphigh/{hr_name}",
                "SEN2NAIP_100_Percent"
            ])

# 7. Package into a SINGLE zip archive
print("Compressing dataset into a single zip file...")
zip_output_path = "/content/SEN2NAIP_Full_Dataset"
shutil.make_archive(zip_output_path, 'zip', local_base)

# 8. Move single zip file to Google Drive (Takes ~10 seconds)
print("Transferring single zip archive to Google Drive...")
shutil.move(f"{zip_output_path}.zip", os.path.join(drive_target_folder, "SEN2NAIP_Full_Dataset.zip"))

# 9. Cleanup local temporary files
print("Cleaning up temporary local files...")
shutil.rmtree(extract_path)
shutil.rmtree(local_base)
os.remove(zip_path)

print(f"Success! Complete dataset saved to Drive as: {drive_target_folder}/SEN2NAIP_Full_Dataset.zip")

In [ ]:
import os

# Check metadata folder contents
metadata_path = '/content/drive/MyDrive/Project_Folder/Datasets/metadata'
print("Metadata files:", os.listdir(metadata_path) if os.path.exists(metadata_path) else "Path does not exist")

# Check Datasets folder contents
datasets_path = '/content/drive/MyDrive/Project_Folder/Datasets'
print("Datasets contents:", os.listdir(datasets_path) if os.path.exists(datasets_path) else "Path does not exist")

Metadata files: ['sen2naip_metadata.csv']
Datasets contents: ['worldstrat_low', 'worldstrat_high', 'metadata', 'sentinaiplow', 'sentinaiphigh', 'SEN2NAIP_Full_Dataset.zip']
